In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Introduction to DataFrames**


## Objectives


A DataFrame is two-dimensional. Columns can be of different data types. DataFrames accept many data inputs including series and other DataFrames. You can pass indexes (row labels) and columns (column labels). Indexes can be numbers, dates, or strings/tuples.

After completing this notebook you will be able to:


* Load a data file into a DataFrame
* View the data schema of a DataFrame
* Perform basic data manipulation
* Aggregate data in a DataFrame


----


## Setup


For this lab, we are going to be using Python and Spark (PySpark).


In [1]:
# Installing required packages
!pip install pyspark
!pip install findspark
!pip install pandas

In [2]:
!apt-get install openjdk-11-jdk -y
!pip install pyspark

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk-headless openjdk-11-jre openjdk-11-jre-headless
  session-migration x11-utils
Suggested packages:
  libxt-doc openjdk-11-demo openjdk-11-source visualvm libnss-mdns
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk openjdk-11-jdk-headless openjdk-

In [3]:
import findspark
findspark.init()

In [4]:
import pandas as pd
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

## Exercise 1 -  Spark session


In this exercise, you will create and initialize the Spark session needed to load the dataframes and operate on it


#### Task 1: Creating the spark session and context


In [5]:
# Creating a spark context class
sc = SparkContext()

# Creating a spark session
spark = SparkSession \
    .builder \
    .appName("Exmaple2") \
    .config("spark.some.config.option", "some-value") \
    .getOrCreate()

#### Task 2: Initialize Spark session
To work with dataframes we just need to verify that the spark session instance has been created.


In [6]:
spark

## Exercise 2 - Load the data and Spark dataframe


In this section, you will first read the CSV file into a Pandas DataFrame and then read it into a Spark DataFrame.
Pandas is a library used for data manipulation and analysis. Pandas offers data structures and operations for creating and manipulating Data Series and DataFrame objects. Data can be imported from various data sources, e.g., Numpy arrays, Python dictionaries, and CSV files. Pandas allows you to manipulate, organize and display the data.
To create a Spark DataFrame we load an external DataFrame, called mtcars. This DataFrame includes 32 observations on 12 variables:


| colIndex | colName | units/description |
| :---: | :--- | :--- |
|[, 1] | mpg |Miles per gallon  |
|[, 2] | cyl | Number of cylinders  |
|[, 3] | disp | Displacement (cu.in.) |  
|[, 4] | hp  | Gross horsepower  |
|[, 5] | drat | Rear axle ratio  |
|[, 6] | wt | Weight (lb/1000)  |
|[, 7] | qsec | 1/4 mile time  |
|[, 8] | vs  | V/S  |
|[, 9] | am | Transmission (0 = automatic, 1 = manual)  |
|[,10] | gear | Number of forward gears  |
|[,11] | carb | Number of carburetors |


#### Task 1: Loading data into a Spark DataFrame


In [7]:
#sdf = spark.read.csv("/content/drive/MyDrive/Colab Notebooks/BigData/Jupyter Notebook/Spark/mtcars.csv", header = True).cache()
sdf = spark.read.csv("mt_cars.csv", header = True).cache()


In [8]:
sdf

DataFrame[Unnamed: 0: string, mpg: string, cyl: string, disp: string, hp: string, drat: string, wt: string, qsec: string, vs: string, am: string, gear: string, carb: string]

In [9]:
# Let us look at the schema of the loaded spark dataframe
sdf.printSchema()

root
 |-- Unnamed: 0: string (nullable = true)
 |-- mpg: string (nullable = true)
 |-- cyl: string (nullable = true)
 |-- disp: string (nullable = true)
 |-- hp: string (nullable = true)
 |-- drat: string (nullable = true)
 |-- wt: string (nullable = true)
 |-- qsec: string (nullable = true)
 |-- vs: string (nullable = true)
 |-- am: string (nullable = true)
 |-- gear: string (nullable = true)
 |-- carb: string (nullable = true)



## Exercise 3: Basic data analysis and manipulation

In this section, we perform basic data analysis and manipulation. We start with previewing the data and then applying some filtering and columwise operations.


#### Task 1: Displays the content of the DataFrame

We use the `show()` method for this. Here we preview the first 5 records. Compare it to a similar `head()` function in Pandas.


In [10]:
sdf.show(5)

+-----------------+----+---+----+---+----+-----+-----+---+---+----+----+
|       Unnamed: 0| mpg|cyl|disp| hp|drat|   wt| qsec| vs| am|gear|carb|
+-----------------+----+---+----+---+----+-----+-----+---+---+----+----+
|        Mazda RX4|  21|  6| 160|110| 3.9| 2.62|16.46|  0|  1|   4|   4|
|    Mazda RX4 Wag|  21|  6| 160|110| 3.9|2.875|17.02|  0|  1|   4|   4|
|       Datsun 710|22.8|  4| 108| 93|3.85| 2.32|18.61|  1|  1|   4|   1|
|   Hornet 4 Drive|21.4|  6| 258|110|3.08|3.215|19.44|  1|  0|   3|   1|
|Hornet Sportabout|18.7|  8| 360|175|3.15| 3.44|17.02|  0|  0|   3|   2|
+-----------------+----+---+----+---+----+-----+-----+---+---+----+----+
only showing top 5 rows


We use the `select()` function to select a particular column of data. Here we show the `mpg` column.


In [11]:
sdf.select('mpg').show(10)

+----+
| mpg|
+----+
|  21|
|  21|
|22.8|
|21.4|
|18.7|
|18.1|
|14.3|
|24.4|
|22.8|
|19.2|
+----+
only showing top 10 rows


#### Task 2: Filtering and Columnar operations

Filtering and Column operations are important to select relevant data and apply useful transformations.


We first filter to only retain rows with mpg > 18. We use the `filter()` function for this.


In [15]:
from pyspark.sql.functions import col
sdf.filter(col("mpg").cast("double") < 18).show()

+-------------------+----+---+-----+---+----+-----+-----+---+---+----+----+
|         Unnamed: 0| mpg|cyl| disp| hp|drat|   wt| qsec| vs| am|gear|carb|
+-------------------+----+---+-----+---+----+-----+-----+---+---+----+----+
|         Duster 360|14.3|  8|  360|245|3.21| 3.57|15.84|  0|  0|   3|   4|
|          Merc 280C|17.8|  6|167.6|123|3.92| 3.44| 18.9|  1|  0|   4|   4|
|         Merc 450SE|16.4|  8|275.8|180|3.07| 4.07| 17.4|  0|  0|   3|   3|
|         Merc 450SL|17.3|  8|275.8|180|3.07| 3.73| 17.6|  0|  0|   3|   3|
|        Merc 450SLC|15.2|  8|275.8|180|3.07| 3.78|   18|  0|  0|   3|   3|
| Cadillac Fleetwood|10.4|  8|  472|205|2.93| 5.25|17.98|  0|  0|   3|   4|
|Lincoln Continental|10.4|  8|  460|215|   3|5.424|17.82|  0|  0|   3|   4|
|  Chrysler Imperial|14.7|  8|  440|230|3.23|5.345|17.42|  0|  0|   3|   4|
|   Dodge Challenger|15.5|  8|  318|150|2.76| 3.52|16.87|  0|  0|   3|   2|
|        AMC Javelin|15.2|  8|  304|150|3.15|3.435| 17.3|  0|  0|   3|   2|
|         Ca

Operating on Columns

Spark also provides a number of functions that can be directly applied to columns for data processing and aggregation. The example below shows the use of basic arithmetic functions to convert the weight values from `lb` to `metric ton`.
We create a new column called `wtTon` that has the weight from the `wt` column converted to metric tons.


In [13]:
sdf.show(5)

+-----------------+----+---+----+---+----+-----+-----+---+---+----+----+
|       Unnamed: 0| mpg|cyl|disp| hp|drat|   wt| qsec| vs| am|gear|carb|
+-----------------+----+---+----+---+----+-----+-----+---+---+----+----+
|        Mazda RX4|  21|  6| 160|110| 3.9| 2.62|16.46|  0|  1|   4|   4|
|    Mazda RX4 Wag|  21|  6| 160|110| 3.9|2.875|17.02|  0|  1|   4|   4|
|       Datsun 710|22.8|  4| 108| 93|3.85| 2.32|18.61|  1|  1|   4|   1|
|   Hornet 4 Drive|21.4|  6| 258|110|3.08|3.215|19.44|  1|  0|   3|   1|
|Hornet Sportabout|18.7|  8| 360|175|3.15| 3.44|17.02|  0|  0|   3|   2|
+-----------------+----+---+----+---+----+-----+-----+---+---+----+----+
only showing top 5 rows


In [16]:
sdf_new = sdf.withColumn('wtTon', sdf['wt'] * 0.45)

In [17]:
sdf_new.show(5)

+-----------------+----+---+----+---+----+-----+-----+---+---+----+----+-------+
|       Unnamed: 0| mpg|cyl|disp| hp|drat|   wt| qsec| vs| am|gear|carb|  wtTon|
+-----------------+----+---+----+---+----+-----+-----+---+---+----+----+-------+
|        Mazda RX4|  21|  6| 160|110| 3.9| 2.62|16.46|  0|  1|   4|   4|  1.179|
|    Mazda RX4 Wag|  21|  6| 160|110| 3.9|2.875|17.02|  0|  1|   4|   4|1.29375|
|       Datsun 710|22.8|  4| 108| 93|3.85| 2.32|18.61|  1|  1|   4|   1|  1.044|
|   Hornet 4 Drive|21.4|  6| 258|110|3.08|3.215|19.44|  1|  0|   3|   1|1.44675|
|Hornet Sportabout|18.7|  8| 360|175|3.15| 3.44|17.02|  0|  0|   3|   2|  1.548|
+-----------------+----+---+----+---+----+-----+-----+---+---+----+----+-------+
only showing top 5 rows


## Exercise 4: Grouping and Aggregation

Spark DataFrames support a number of commonly used functions to aggregate data after grouping. In this example we compute the average weight of cars by their cylinders as shown below.


In [18]:
sdf_new.show(10)

+-----------------+----+---+-----+---+----+-----+-----+---+---+----+----+-------+
|       Unnamed: 0| mpg|cyl| disp| hp|drat|   wt| qsec| vs| am|gear|carb|  wtTon|
+-----------------+----+---+-----+---+----+-----+-----+---+---+----+----+-------+
|        Mazda RX4|  21|  6|  160|110| 3.9| 2.62|16.46|  0|  1|   4|   4|  1.179|
|    Mazda RX4 Wag|  21|  6|  160|110| 3.9|2.875|17.02|  0|  1|   4|   4|1.29375|
|       Datsun 710|22.8|  4|  108| 93|3.85| 2.32|18.61|  1|  1|   4|   1|  1.044|
|   Hornet 4 Drive|21.4|  6|  258|110|3.08|3.215|19.44|  1|  0|   3|   1|1.44675|
|Hornet Sportabout|18.7|  8|  360|175|3.15| 3.44|17.02|  0|  0|   3|   2|  1.548|
|          Valiant|18.1|  6|  225|105|2.76| 3.46|20.22|  1|  0|   3|   1|  1.557|
|       Duster 360|14.3|  8|  360|245|3.21| 3.57|15.84|  0|  0|   3|   4| 1.6065|
|        Merc 240D|24.4|  4|146.7| 62|3.69| 3.19|   20|  1|  0|   4|   2| 1.4355|
|         Merc 230|22.8|  4|140.8| 95|3.92| 3.15| 22.9|  1|  0|   4|   2| 1.4175|
|         Merc 2

In [19]:
sdf.groupby(['cyl']).agg({"wt": "AVG"}).show(5)

+---+------------------+
|cyl|           avg(wt)|
+---+------------------+
|  8|3.9992142857142867|
|  6| 3.117142857142857|
|  4| 2.285727272727273|
+---+------------------+



In [20]:
sdf_new.groupby(['gear']).agg({"wt": "AVG"}).show(5)

+----+------------------+
|gear|           avg(wt)|
+----+------------------+
|   3|            3.8926|
|   5|            2.6326|
|   4|2.6166666666666667|
+----+------------------+



In [21]:
3.8926*0.45

1.75167

In [22]:
sdf_new.groupby(['gear']).agg({"wtTon": "AVG"}).show(5)

+----+------------------+
|gear|        avg(wtTon)|
+----+------------------+
|   3|1.7516700000000003|
|   5|1.1846700000000001|
|   4|            1.1775|
+----+------------------+



We can also sort the output from the aggregation to get the most common cars.


In [23]:
car_counts = sdf.groupby(['cyl']).agg({"wt": "count"}).show(5)


+---+---------+
|cyl|count(wt)|
+---+---------+
|  8|       14|
|  6|        7|
|  4|       11|
+---+---------+



## Practice Questions


### Question 1 - DataFrame basics


Display the first 5 rows of all cars that have atleast 5 cylinders.


In [ ]:
sdf.filter(sdf['cyl'] >= 5).show()

+-------------------+----+---+-----+---+----+-----+-----+---+---+----+----+
|         Unnamed: 0| mpg|cyl| disp| hp|drat|   wt| qsec| vs| am|gear|carb|
+-------------------+----+---+-----+---+----+-----+-----+---+---+----+----+
|          Mazda RX4|  21|  6|  160|110| 3.9| 2.62|16.46|  0|  1|   4|   4|
|      Mazda RX4 Wag|  21|  6|  160|110| 3.9|2.875|17.02|  0|  1|   4|   4|
|     Hornet 4 Drive|21.4|  6|  258|110|3.08|3.215|19.44|  1|  0|   3|   1|
|  Hornet Sportabout|18.7|  8|  360|175|3.15| 3.44|17.02|  0|  0|   3|   2|
|            Valiant|18.1|  6|  225|105|2.76| 3.46|20.22|  1|  0|   3|   1|
|         Duster 360|14.3|  8|  360|245|3.21| 3.57|15.84|  0|  0|   3|   4|
|           Merc 280|19.2|  6|167.6|123|3.92| 3.44| 18.3|  1|  0|   4|   4|
|          Merc 280C|17.8|  6|167.6|123|3.92| 3.44| 18.9|  1|  0|   4|   4|
|         Merc 450SE|16.4|  8|275.8|180|3.07| 4.07| 17.4|  0|  0|   3|   3|
|         Merc 450SL|17.3|  8|275.8|180|3.07| 3.73| 17.6|  0|  0|   3|   3|
|        Mer

### Question 2 - DataFrame aggregation


Using the functions and tables shown above, print out the mean weight of a car in our database in metric tons.


In [ ]:
sdf_new.groupby(['Unnamed: 0']).agg({"wtTon": "AVG"}).show()

+------------------+------------------+
|        Unnamed: 0|        avg(wtTon)|
+------------------+------------------+
|    Ford Pantera L|            1.4265|
|        Merc 450SL|            1.6785|
|       Honda Civic|           0.72675|
|      Ferrari Dino|            1.2465|
|     Mazda RX4 Wag|           1.29375|
|         Mazda RX4|             1.179|
|         Merc 240D|            1.4355|
|       Merc 450SLC|1.7009999999999998|
|         Fiat X1-9|           0.87075|
|  Pontiac Firebird|           1.73025|
|        Merc 450SE|1.8315000000000001|
| Chrysler Imperial|           2.40525|
|Cadillac Fleetwood|2.3625000000000003|
|        Volvo 142E|             1.251|
|        Duster 360|            1.6065|
|          Merc 230|            1.4175|
|          Fiat 128|0.9900000000000001|
|         Merc 280C|             1.548|
|      Lotus Europa|           0.68085|
|       AMC Javelin|           1.54575|
+------------------+------------------+
only showing top 20 rows



### Question 3 - DataFrame columnar operations


In the earlier sections of this notebook, we have created a new column called `wtTon` to indicate the weight in metric tons using a standard conversion formula. In this case we have applied this directly to the dataframe column `wt` as it is a linear operation (multiply by 0.45). Similarly, as part of this exercise, create a new column for mileage in `kmpl` (kilometer-per-liter) instead of `mpg`(miles-per-gallon) by using a conversion factor of 0.425.

Additionally sort the output in decreasing order of mileage in kmpl.


In [ ]:
from pyspark.sql.functions import round
sdf_newest = sdf_new.withColumn('kmpl',round(sdf_new['mpg'] * 0.425))

In [ ]:
sdf_newest.show()

+-------------------+----+---+-----+---+----+-----+-----+---+---+----+----+------------------+----+
|         Unnamed: 0| mpg|cyl| disp| hp|drat|   wt| qsec| vs| am|gear|carb|             wtTon|kmpl|
+-------------------+----+---+-----+---+----+-----+-----+---+---+----+----+------------------+----+
|          Mazda RX4|  21|  6|  160|110| 3.9| 2.62|16.46|  0|  1|   4|   4|             1.179| 9.0|
|      Mazda RX4 Wag|  21|  6|  160|110| 3.9|2.875|17.02|  0|  1|   4|   4|           1.29375| 9.0|
|         Datsun 710|22.8|  4|  108| 93|3.85| 2.32|18.61|  1|  1|   4|   1|             1.044|10.0|
|     Hornet 4 Drive|21.4|  6|  258|110|3.08|3.215|19.44|  1|  0|   3|   1|           1.44675| 9.0|
|  Hornet Sportabout|18.7|  8|  360|175|3.15| 3.44|17.02|  0|  0|   3|   2|             1.548| 8.0|
|            Valiant|18.1|  6|  225|105|2.76| 3.46|20.22|  1|  0|   3|   1|             1.557| 8.0|
|         Duster 360|14.3|  8|  360|245|3.21| 3.57|15.84|  0|  0|   3|   4|            1.6065| 6.0|


In [ ]:
spark.stop()

# Exercise: nycflights13 Dataset

In [ ]:
!pip install nycflights13

from nycflights13 import flights as flights_pd, airlines as airlines_pd, airports as airports_pd, weather as weather_pd

from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder.appName("NYCFlights13").getOrCreate()

# Convert each dataset to Spark DataFrame
flights = spark.createDataFrame(flights_pd)
airlines = spark.createDataFrame(airlines_pd)
airports = spark.createDataFrame(airports_pd)
weather = spark.createDataFrame(weather_pd)

print("Spark DataFrames for flights, airlines, airports, and weather have been created.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 30.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for nycflights13: filename=nycflights13-0.0.3-py3-none-any.whl size=8732722 sha256=38693d300af5870903661af3fb858d76c9366efe0a99971a2e86b8ef07db4150
  Stored in directory: /root/.cache/pip/wheels/58/ee/20/7838832df5510701439239ec10c2d8927de32aac2729ce2085
Successfully built nycflights13
Spark DataFrames for flights, airlines, airports, and weather have been created.


In [ ]:
flights.createOrReplaceTempView("flights")
airlines.createOrReplaceTempView("airlines")
airports.createOrReplaceTempView("airports")
weather.createOrReplaceTempView("weather")

print("Temporary views created for flights, airlines, airports, and weather.")

Temporary views created for flights, airlines, airports, and weather.


In [ ]:
flights.show(5)

+----+-----+---+--------+--------------+---------+--------+--------------+---------+-------+------+-------+------+----+--------+--------+----+------+--------------------+
|year|month|day|dep_time|sched_dep_time|dep_delay|arr_time|sched_arr_time|arr_delay|carrier|flight|tailnum|origin|dest|air_time|distance|hour|minute|           time_hour|
+----+-----+---+--------+--------------+---------+--------+--------------+---------+-------+------+-------+------+----+--------+--------+----+------+--------------------+
|2013|    1|  1|   517.0|           515|      2.0|   830.0|           819|     11.0|     UA|  1545| N14228|   EWR| IAH|   227.0|    1400|   5|    15|2013-01-01T10:00:00Z|
|2013|    1|  1|   533.0|           529|      4.0|   850.0|           830|     20.0|     UA|  1714| N24211|   LGA| IAH|   227.0|    1416|   5|    29|2013-01-01T10:00:00Z|
|2013|    1|  1|   542.0|           540|      2.0|   923.0|           850|     33.0|     AA|  1141| N619AA|   JFK| MIA|   160.0|    1089|   5|   